# Token 生命周期与成本演示

这个 notebook 不是交易策略，而是一个可复现的分析模板：

- 从公开样本中读取 `pump_fun` 与 `pump_amm` 的 parquet 文件。
- 找到样本里出现频率较高的 token。
- 按时间整理它的原始事件序列。
- 用一个简单成本模型估算：买入后至少涨多少才可能回本。

完整 release 数据下载后，也可以沿用同样思路，把 `SAMPLE_DIR` 换成完整数据目录。


In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
SAMPLE_DIR = ROOT / "data_sample"
assert SAMPLE_DIR.exists(), SAMPLE_DIR


## 1. 读取公开样本

`data_sample/` 只包含小样本，适合快速演示。它不是完整市场，也不应该用于下结论。


In [ ]:
frames = []
for path in sorted(SAMPLE_DIR.glob("*.parquet")):
    table = pq.read_table(path)
    df = table.to_pandas()
    df["source_file"] = path.name
    frames.append(df)

events = pd.concat(frames, ignore_index=True)
events.shape, events.columns.tolist()[:12]


## 2. 找一个样本 token

这里选择样本中事件数最多的 `token_mint`。这只是为了演示生命周期查看方式，不代表它更值得交易。


In [ ]:
token_counts = events["token_mint"].value_counts().head(10)
token_counts


In [ ]:
target_token = token_counts.index[0]
token_events = events[events["token_mint"] == target_token].copy()

if "timestamp" in token_events.columns:
    token_events["event_time"] = pd.to_datetime(token_events["timestamp"], unit="s", errors="coerce")
    token_events = token_events.sort_values("event_time")

target_token, token_events.shape


## 3. 查看事件序列

优先关注确认时能看到的字段，例如时间、方向、数量、SOL 金额、签名和来源文件。不要把未来最高点当成入场时已知信息。


In [ ]:
visible_cols = [
    c for c in [
        "event_time", "action", "token_amount", "lamports_amount",
        "fee_lamports", "user_wallet", "signature", "source_file"
    ] if c in token_events.columns
]
token_events[visible_cols].head(20)


## 4. 简单成本模型

真实交易里，回本不只取决于价格涨幅，还取决于平台费、协议费、滑点、优先费、网络费和失败交易成本。下面只是一个保守的玩具模型，方便建立成本意识。


In [ ]:
def break_even_return(
    buy_sol=0.10,
    buy_fee_rate=0.01,
    sell_fee_rate=0.01,
    slippage_rate=0.015,
    priority_fee_sol=0.0005,
    network_fee_sol=0.00001,
    failed_tx_cost_sol=0.0,
):
    fixed_cost = priority_fee_sol * 2 + network_fee_sol * 2 + failed_tx_cost_sol
    rate_cost = buy_fee_rate + sell_fee_rate + slippage_rate
    total_cost_sol = buy_sol * rate_cost + fixed_cost
    return {
        "buy_sol": buy_sol,
        "rate_cost_pct": rate_cost * 100,
        "fixed_cost_sol": fixed_cost,
        "total_cost_sol": total_cost_sol,
        "break_even_pct": total_cost_sol / buy_sol * 100,
    }

pd.DataFrame([
    break_even_return(buy_sol=0.02),
    break_even_return(buy_sol=0.10),
    break_even_return(buy_sol=0.50),
])


## 5. 下一步

更严肃的研究可以继续扩展：

- 从完整 release 数据读取某个 token 的全生命周期。
- 对比 `pump_fun` 内盘阶段与 `pump_amm` 外盘阶段。
- 用实际成交序列估算滑点和价格冲击。
- 把成本模型前置到任何回测报告里，而不是最后才扣。 
